In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from histokit.io.annotations.annotation import AnnotationSet
from histokit.io.slides.slide import SlideBase
from histokit.segmentation.detectors import per_patch_canny_ranker, per_patch_canny_segmentation
from histokit.utils.geometry import Shape

def visualise_detection(
    slide: SlideBase,
    patch_size: int,
    patch_level: int,
    annotation_set: AnnotationSet | None = None,
    alpha: float = 0.3,
) -> None:
    """Visualise canny edge detection alongside the slide thumbnail and annotations.

    The thumbnail dimensions are derived from the patch grid so that each pixel
    in the mask corresponds to one pixel in the thumbnail.

    Args:
        slide: An open SlideBase instance.
        patch_size: Size of each patch in pixels at *patch_level*.
        patch_level: Pyramid level at which patches are read.
        annotation_set: Optional AnnotationSet to overlay on the thumbnail.
        alpha: Blending factor for annotation overlay.
    """
    # Build and run both canny edge detectors
    ranker = per_patch_canny_ranker(patch_size=patch_size, patch_level=patch_level)
    ranker_mask = ranker(slide)

    segmenter = per_patch_canny_segmentation(patch_size=patch_size, patch_level=patch_level)
    seg_mask = segmenter(slide)

    # Derive thumbnail size from the mask grid
    thumb_w, thumb_h = ranker_mask.shape[1], ranker_mask.shape[0]
    thumb = slide.get_thumbnail_for_size(thumb_w, thumb_h)

    ncols = 3 if annotation_set is None else 4
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 7))

    # Thumbnail
    axes[0].imshow(thumb)
    axes[0].set_title("Slide thumbnail")
    axes[0].axis("off")

    # Edge density (ranker)
    axes[1].imshow(ranker_mask, cmap="hot")
    axes[1].set_title(ranker.name)
    axes[1].axis("off")

    # Binary segmentation
    axes[2].imshow(seg_mask, cmap="gray")
    axes[2].set_title(segmenter.name)
    axes[2].axis("off")

    # Annotation overlay
    if annotation_set is not None:
        slide_w, slide_h = slide.dimensions[0]
        factor = ((slide_w / thumb_w) + (slide_h / thumb_h)) / 2

        ann_mask = annotation_set.render(Shape(thumb_h, thumb_w), factor)

        fill_val = annotation_set.label_map[annotation_set.fill_label]
        label_values = sorted(annotation_set.label_map.values())
        cmap = plt.get_cmap("tab10", len(label_values))
        colours = {v: (np.array(cmap(i)[:3]) * 255).astype(np.uint8) for i, v in enumerate(label_values)}

        overlay = thumb.copy()
        for label_val, colour in colours.items():
            if label_val == fill_val:
                continue
            region = ann_mask == label_val
            if region.any():
                overlay[region] = colour

        blended = (thumb.astype(float) * (1 - alpha) + overlay.astype(float) * alpha).astype(np.uint8)
        axes[3].imshow(blended)
        axes[3].set_title("Annotations")
        axes[3].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path

from histokit.io.slides import get_slide_cls_for_path
from histokit.io.annotations import get_annotation_loader_for_path, AnnotationSet
from histokit.dataset.schema import DatasetSchema

slide_path = Path("../data/icaird/cervical_mini/slides/example-slide-001.ome.tiff")
annotations_path = Path("../data/icaird/cervical_mini/annotations/example-slide-001.geojson")
labels_path = Path("../data/icaird/cervical_mini/labels.json")

# Load annotations
schema = DatasetSchema.from_json(labels_path).annotations
load_annotation = get_annotation_loader_for_path(annotations_path)
annotation_regions = load_annotation(annotations_path, schema)
annotation_set = AnnotationSet(annotation_regions, schema)

# Load slide and visualise
SlideCls = get_slide_cls_for_path(slide_path)
with SlideCls(slide_path) as slide:
    visualise_detection(slide, patch_size=224, patch_level=1, annotation_set=annotation_set)

In [ ]:
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

from histokit.io.slides import get_slide_cls_for_path
from histokit.io.annotations import get_annotation_loader_for_path, AnnotationSet
from histokit.dataset.schema import DatasetSchema

slide_path = Path("../data/icaird/cervical_mini/slides/example-slide-001.ome.tiff")
annotations_path = Path("../data/icaird/cervical_mini/annotations/example-slide-001.geojson")
labels_path = Path("../data/icaird/cervical_mini/labels.json")

# Load annotations
schema = DatasetSchema.from_json(labels_path).annotations
load_annotation = get_annotation_loader_for_path(annotations_path)
annotation_regions = load_annotation(annotations_path, schema)
annotation_set = AnnotationSet(annotation_regions, schema)

SCALE = 8  # each patch-grid cell becomes 8x8 pixels in the visualisation

# Bold, distinct colours keyed by label name — easy to tell apart on H&E
ANNOTATION_COLOUR_MAP = {
    "background":           np.array([180, 180, 180], dtype=np.uint8),  # light grey
    "normal_inflammation":  np.array([  0, 200, 255], dtype=np.uint8),  # sky blue
    "low_grade":            np.array([255, 255,   0], dtype=np.uint8),  # bright yellow
    "high_grade":           np.array([255, 140,   0], dtype=np.uint8),  # orange
    "malignant":            np.array([255,   0,   0], dtype=np.uint8),  # red
}

SlideCls = get_slide_cls_for_path(slide_path)
with SlideCls(slide_path) as slide:
    # Run detectors at patch-grid resolution (224px patches, level 1)
    ranker = per_patch_canny_ranker(patch_size=224, patch_level=1)
    ranker_mask = ranker(slide)

    segmenter = per_patch_canny_segmentation(patch_size=224, patch_level=1)
    seg_mask = segmenter(slide)

    # Render annotations at the same patch-grid resolution
    grid_h, grid_w = ranker_mask.shape
    slide_w, slide_h = slide.dimensions[0]
    factor = ((slide_w / grid_w) + (slide_h / grid_h)) / 2
    ann_mask = annotation_set.render(Shape(grid_h, grid_w), factor)

    # Upscale everything to the visualisation size
    vis_h, vis_w = grid_h * SCALE, grid_w * SCALE
    ranker_vis = np.array(Image.fromarray(ranker_mask.astype(np.float32)).resize((vis_w, vis_h), Image.Resampling.NEAREST))
    seg_vis = np.array(Image.fromarray(seg_mask.astype(np.float32)).resize((vis_w, vis_h), Image.Resampling.NEAREST))
    ann_vis = np.array(Image.fromarray(ann_mask.astype(np.float32)).resize((vis_w, vis_h), Image.Resampling.NEAREST))

    # Get a thumbnail at the upscaled size
    thumb = slide.get_thumbnail_for_size(vis_w, vis_h)

    # Build annotation overlay
    fill_val = annotation_set.label_map[annotation_set.fill_label]
    val_to_name = {v: k for k, v in annotation_set.label_map.items()}
    colours = {}
    for label_name, label_val in annotation_set.label_map.items():
        if label_val == fill_val:
            continue
        if label_name in ANNOTATION_COLOUR_MAP:
            colours[label_val] = ANNOTATION_COLOUR_MAP[label_name]

    alpha = 0.55
    overlay = thumb.copy()
    for label_val, colour in colours.items():
        region = ann_vis == label_val
        if region.any():
            overlay[region] = colour

    blended = (thumb.astype(float) * (1 - alpha) + overlay.astype(float) * alpha).astype(np.uint8)

    # Build masked version: darken background patches (semi-transparent black)
    bg_mask = seg_vis == 0
    masked = blended.copy().astype(float)
    masked[bg_mask] = masked[bg_mask] * 0.5  # dim to 50% brightness
    masked = masked.astype(np.uint8)

    fig, axes = plt.subplots(2, 2, figsize=(30, 36))

    TITLE_FONT_SIZE = 32

    axes[0, 0].imshow(thumb)
    axes[0, 0].set_title("Slide Thumbnail", fontsize=TITLE_FONT_SIZE)
    axes[0, 0].axis("off")

    # PowerNorm with gamma < 1 stretches low values across more of the colormap
    norm = mcolors.PowerNorm(gamma=0.3, vmin=0.0, vmax=1.0)
    im = axes[0, 1].imshow(ranker_vis, cmap="hot", norm=norm)
    axes[0, 1].set_title("Fraction of Pixels that are Edges (0.0 to 1.0) Per Patch", fontsize=TITLE_FONT_SIZE)
    axes[0, 1].axis("off")
    # Colorbar sized to match the image height
    divider = make_axes_locatable(axes[0, 1])
    cax = divider.append_axes("right", size="3%", pad=0.15)
    fig.colorbar(im, cax=cax)
    cax.tick_params(labelsize=24)

    axes[1, 0].imshow(blended)
    axes[1, 0].set_title("Annotations", fontsize=TITLE_FONT_SIZE)
    axes[1, 0].axis("off")

    axes[1, 1].imshow(masked)
    axes[1, 1].set_title("Background Segmentation over Annotations", fontsize=TITLE_FONT_SIZE)
    axes[1, 1].axis("off")

    # Build legend patches for annotation labels + background
    legend_handles = []
    for label_val, colour in colours.items():
        name = val_to_name.get(label_val, f"Label {label_val}").replace("_", " ").title()
        legend_handles.append(mpatches.Patch(color=colour / 255.0, label=name))
    legend_handles.append(mpatches.Patch(color=(0.125, 0.125, 0.125), label="Background (Segmentation)"))

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(legend_handles),
        fontsize=24,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02),
    )

    plt.tight_layout(h_pad=6, rect=[0, 0.04, 1, 1])
    plt.show()